In [1]:
import sys
# TO CHANGE
BASEDIR = "../../../.."
sys.path.insert(0, BASEDIR)

In [2]:
from pprint import pprint

In [3]:
from src.kg_model import KnowledgeGraphModel, KnowledgeGraphModelConfig
from src.db_drivers.vector_driver import EmbedderModelConfig
from src.pipelines.memorize import MemPipelineConfig, MemPipeline, LLMExtractorConfig, LLMUpdatorConfig

/home/dzigen/Desktop/Projects/PersonalAI/.pai_venv/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


## Пример работы с Memorize-пайплайном (по построению графа знаний)

1. Инициализация модели графа знаний

In [4]:
kg_config = KnowledgeGraphModelConfig(
    nodestree_config=None # Модель дерева вершин строить не будет
)

EMBEDDER_MODEL_PATH = '../../../../models/intfloat/multilingual-e5-small' # PATH TO APPROPRIATE EMBEDDER-MODEL
kg_config.embedders_config['m-e5-small'] = EmbedderModelConfig(model_name_or_path=EMBEDDER_MODEL_PATH)


kg_model = KnowledgeGraphModel(kg_config)

No sentence-transformers model found with name ../../../../models/intfloat/multilingual-e5-small. Creating a new one with mean pooling.


2. Инициализация Memorize-пайплайна

In [5]:
mem_config = MemPipelineConfig(
    extractor_config=LLMExtractorConfig(lang='ru'),
    updator_config=LLMUpdatorConfig(
        lang='ru', 
        delete_obsolete_info=False # Выключаем механизм по поиску/удалению устревших знаний в графе при добавлении новой информации
    )
)

In [6]:
mem_pipeline = MemPipeline(kg_model, mem_config)

3. Формируем триплеты (из входных текстовых фрагментов на естественном языке) и добавляем в граф знаний

In [7]:
TEXT_EXAMPLE1 = "Шла саша по шоссе."
TEXT_EXAMPLE2 = "Шла маша по шоссе."
TEXT_EXAMPLE3 = "Теплоход плыл по водному каналу."
TEXT_EXAMPLE4 = "Моторная лодка плыла по реке."

In [10]:
print("\n1.Входной текст: ", TEXT_EXAMPLE1)
extracted_triplets, rinfo = mem_pipeline.remember(TEXT_EXAMPLE1)
print("Извлечённые из текста триплеты:")
pprint(extracted_triplets, width=200)
print("Информация о завершении операции: ", rinfo)

print("\n2.Входной текст: ", TEXT_EXAMPLE2)
extracted_triplets, rinfo = mem_pipeline.remember(TEXT_EXAMPLE2)
print("Извлечённые из текста триплеты:")
pprint(extracted_triplets, width=200)
print("Информация о завершении операции: ", rinfo)

print("\n3.Входной текст: ", TEXT_EXAMPLE3)
extracted_triplets, rinfo = mem_pipeline.remember(TEXT_EXAMPLE3)
print("Извлечённые из текста триплеты:")
pprint(extracted_triplets, width=200)
print("Информация о завершении операции: ", rinfo)

print("\n4.Входной текст: ", TEXT_EXAMPLE4)
extracted_triplets, rinfo = mem_pipeline.remember(TEXT_EXAMPLE4)
print("Извлечённые из текста триплеты:")
pprint(extracted_triplets, width=200)
print("Информация о завершении операции: ", rinfo)



1.Входной текст:  Шла саша по шоссе.
Извлечённые из текста триплеты:
[Triplet(start_node=Node(name='саша', type=<NodeType.object: 'object'>, prop={}, stringified='саша', id='72adc4fc2f53b1344fb5b56d6ed81b1f'),
         relation=Relation(name='проходит', type=<RelationType.simple: 'simple'>, prop={}, id='da0b488d41d6458245fac501fc2e5a0b'),
         end_node=Node(name='шоссе', type=<NodeType.object: 'object'>, prop={}, stringified='шоссе', id='956c2d5e231f655cee06efa6a32969e9'),
         stringified='саша проходит шоссе',
         id='017d461c6c38d3260ca906ac65953228'),
 Triplet(start_node=Node(name='саша', type=<NodeType.object: 'object'>, prop={}, stringified='саша', id='72adc4fc2f53b1344fb5b56d6ed81b1f'),
         relation=Relation(name='hyper', type=<RelationType.hyper: 'hyper'>, prop={}, id='b7cb6abc49e154002f814107ef31a20f'),
         end_node=Node(name='1. саша проходит по шоссе', type=<NodeType.hyper: 'hyper'>, prop={}, stringified='1. саша проходит по шоссе', id='b7cb6abc49e154

In [11]:
kg_model.count_items()

{'graph_info': {'triplets': 35, 'nodes': 22},
 'embeddings_info': {'nodes': 22, 'triplets': 12},
 'nodestree_info': None}

In [12]:
del mem_pipeline

In [13]:
kg_model.clear()
del kg_model